In [1]:
import sys
sys.path.append('../')

import scqubits as scq
import pandas as pd
import qutip as qt
import numpy as np
from matplotlib import pyplot as plt
from qutip.qip.operations import rz, cz_gate
import cmath
from tqdm import tqdm
from matplotlib.colors import LogNorm
import datetime
import pytz
import scqubits.settings as settings
settings.OVERLAP_THRESHOLD = 0.3
from joblib import Parallel, delayed
import itertools
import scipy.sparse as ssp
from sympy import symbols
import scipy as sp
import utils_2Q_gate_zp as ut
import os
from datetime import datetime
from multiprocessing import Pool

In [5]:
np.arange(18)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17])

### Noise model

In [2]:
# drive_phi, drive_theta, truc = False, True, 150
drive_phi, drive_theta, truc = True, False, 150
drive_0 = True

t1_other = 170 # μs
gamma_decay_other =  1 / 1e3 / t1_other
gamma_dephase_other = 1 / 1e3 / t1_other

############################################################
folder = 'data_xgate_theta_3ncut.txt' if drive_theta else 'data_xgate_phi_3ncut.txt'
f_xgate = pd.read_csv('data/'+folder)
params = f_xgate[['tg', 'drive_amp_1', 'drive_amp_2', 'detune_1', 'detune_2'
                    ]].to_numpy()[[0], :]

num_cpus, n_job = 4, 1*len(params)
logi_state = [0, 2]
folder = '../../data/3ncut_one_zeropi/'
if drive_0:
    evals = 2*np.pi* scq.read(folder + f'zeropi_0_specdata_truc=1000_3ncut.h5').energy_table
    n_theta = 2*np.pi* scq.read(folder + f'zeropi_0_n_theta_truc=1000_3ncut.h5').matrixelem_table
    n_phi = 2*np.pi* scq.read(folder + f'zeropi_0_n_phi_truc=1000_3ncut.h5').matrixelem_table
else:
    evals = 2*np.pi* scq.read(folder + f'zeropi_1_specdata_truc=1000_3ncut.h5').energy_table
    n_theta = 2*np.pi* scq.read(folder + f'zeropi_1_n_theta_truc=1000_3ncut.h5').matrixelem_table
    n_phi = 2*np.pi* scq.read(folder + f'zeropi_1_n_phi_truc=1000_3ncut.h5').matrixelem_table
evals = evals - evals[0]
H0 = qt.Qobj(np.diag(evals))
if drive_phi:
    w_trans_1 = evals[9] - evals[0]
    w_trans_2 = evals[9] - evals[2]
    drive_term = n_phi
if drive_theta:
    w_trans_1 = evals[7] - evals[0]
    w_trans_2 = evals[7] - evals[2]
    drive_term = n_theta

############################################################
thresh = 0.01
hspace_charge = [0, 2]  # Start with the ground and first excited states
for s in hspace_charge:
    for i in range(truc):
        if np.abs(drive_term[s, i] / (2 * np.pi)) > thresh and i not in hspace_charge:
            hspace_charge.append(i)
hspace_charge.sort()
############################################################
# hspace_charge = np.arange(truc).tolist()

############################################################
hspace_len = len(hspace_charge)
logi_idx = [hspace_charge.index(s) for s in logi_state]
H0_truc = ut.truncate_2(H0, hspace_charge)
drive_truc = ut.truncate_2(drive_term, hspace_charge)
H_qbt_drive = [H0_truc, [drive_truc, ut.drive_gauss_A],
                        [drive_truc, ut.drive_gauss_B],]
############################################################
print('params =')
for para in params:
    print(para.tolist(), ',')
print('hspace_len=', hspace_len)
print(' hspace_charge = [')
for i in range(0, len(hspace_charge), 10):
    print(', '.join(map(str, hspace_charge[i:i+10])), ',')
print(']')

params =
[20.000087, 0.249753, 0.216297, 0.43883, 0.490612] ,
hspace_len= 81
 hspace_charge = [
0, 2, 3, 4, 8, 9, 10, 11, 15, 16 ,
18, 19, 20, 23, 24, 25, 28, 31, 33, 35 ,
37, 39, 40, 42, 43, 46, 47, 49, 51, 53 ,
55, 56, 57, 60, 62, 64, 66, 67, 69, 70 ,
73, 76, 77, 79, 81, 83, 85, 86, 88, 91 ,
92, 95, 96, 98, 100, 101, 102, 104, 106, 108 ,
110, 112, 113, 116, 118, 120, 121, 123, 126, 128 ,
130, 132, 133, 136, 137, 139, 141, 143, 145, 146 ,
148 ,
]


### New model - decay to not just $\ket{0}$

In [3]:
folder_1 = 'data/data_gamma_'
folder_2 = 'theta.txt' if drive_theta else 'phi_truc200.txt'
gamma_new = pd.read_csv(folder_1 + folder_2)
gamma_dephase_new = gamma_new['tphi_50us_02'].to_numpy()

if drive_theta:
    Gamma = gamma_decay_other / (np.abs(n_theta[4,7])**2)
else:
    Gamma = gamma_decay_other / (np.abs(n_phi[4,9])**2)
gamma_decay_new = Gamma* np.abs(drive_truc.full())**2    
gamma_dephase_new = gamma_dephase_new *50 /t1_other
jump_t1   = []
jump_tphi = []
for i in range(1,hspace_len):
    for j in range(0,i):
        jump_t1.append( np.sqrt(gamma_decay_new[i,j]) * qt.basis(hspace_len,j) * qt.basis(hspace_len,i).dag() )
    jump_tphi.append( np.sqrt(2*gamma_dephase_new[i]) * qt.basis(hspace_len,i).proj() )
print('np.shape(jump_t1)=',  np.shape(jump_t1), '; np.shape(jump_tphi)=',  np.shape(jump_tphi)) 

np.shape(jump_t1)= (3240, 81, 81) ; np.shape(jump_tphi)= (80, 81, 81)


### noise simulation

In [4]:
c_op_list = [qt.Qobj(np.zeros((truc, truc)))]
c_op_list = []
args = [H_qbt_drive, w_trans_1, w_trans_2, num_cpus, c_op_list, logi_idx]
f_ideal = Parallel(n_jobs=n_job)(delayed(ut.xgate_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('\nf_ideal = [')
for i in range(0, len(f_ideal), 4):
    print(', '.join(map(str, f_ideal[i:i+4])), ',')
print(']')
print("Current Mountain Time:", datetime.now(pytz.timezone('America/Denver')))

############################################################
c_op_list = jump_t1 + jump_tphi
args = [H_qbt_drive, w_trans_1, w_trans_2, num_cpus, c_op_list, logi_idx]
f_noise = Parallel(n_jobs=n_job)(delayed(ut.xgate_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('\nf_noise = [')
for i in range(0, len(f_noise), 4):
    print(', '.join(map(str, f_noise[i:i+4])), ',')
print(']')
print("Current Mountain Time:", datetime.now(pytz.timezone('America/Denver')))


 len(c_op_list)=0
U_noise.istp=False
U_noise.iscp=True
U_noise=Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = False
Qobj data =
[[-0.09677902+0.11381732j -0.70889609-0.15246055j]
 [-0.69413469-0.09709102j  0.10934676+0.26439215j]]
fidelity= 0.6717952219801969

f_ideal = [
-0.4838551007584577 ,
]
Current Mountain Time: 2025-06-24 23:30:52.199395-06:00

 len(c_op_list)=3320
U_noise.istp=True
U_noise.iscp=False
U_noise=Quantum object: dims = [[[81], [81]], [[2], [2]]], shape = (6561, 4), type = super, isherm = False
Qobj data =
[[ 2.22079256e-02+1.52226377e-16j  5.06830547e-02+9.50951862e-02j
   5.06830547e-02-9.50951862e-02j  5.25141721e-01-6.54774328e-16j]
 [ 5.55471811e-02+8.81942444e-02j  1.95520401e-02-3.78545380e-02j
   5.06150060e-01-3.65757356e-02j -1.17663394e-01-1.70537875e-01j]
 [ 2.23974920e-05-3.82706437e-06j -1.91834418e-05-2.01280156e-05j
   3.36862502e-05-1.07002788e-04j -1.29165851e-04+3.81329237e-05j]
 ...
 [ 2.11476287e-03+1.29467865e-03j -2.0

In [25]:
# print(np.shape(n_theta), np.shape(n_phi), np.shape(evals))


In [ ]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

: 

### Old model - decay to $\ket{0}$

In [ ]:
# ### old model
# idx_2 = 2 if drive_theta else 1

# hspace_len = len(hspace_charge)
# if drive_theta:
#     gamma_decay_old   = [0, gamma_decay_other,  gamma_decay_logi]  + [gamma_decay_other]  * (hspace_len-3)
#     gamma_dephase_old = [0, gamma_dephase_other, gamma_dephase_logi] + [gamma_dephase_other] * (hspace_len-3)
# else:
#     gamma_decay_old   = [0,  gamma_decay_logi]  + [gamma_decay_other]  * (hspace_len-2)
#     gamma_dephase_old = [0,  gamma_dephase_logi] + [gamma_dephase_other] * (hspace_len-2)

# folder_1 = 'data/data_gamma_'
# folder_2 = 'theta.txt' if drive_theta else 'phi.txt'
# gamma_new = pd.read_csv(folder_1 + folder_2)

# ### 't1_50us_47', 't1_50us_27', 't1_50us_07'
# ### 'tphi_50us_02', 'tphi_50us_07', 'tphi_1e6'
# gamma_decay_new = gamma_new['t1_50us_47'].to_numpy()
# # gamma_dephase_new = gamma_new['tphi_1e6'].to_numpy()
# gamma_dephase_new = gamma_new['tphi_50us_02'].to_numpy()

# print("gamma_decay_new[2] = ", gamma_decay_new[2], ", gamma_dephase_new[2] = ", gamma_dephase_new[2])
# gamma_decay_new = gamma_decay_new *50 /t1_other
# gamma_dephase_new = gamma_dephase_new *50 /t1_other

# jump_t1   = []
# jump_tphi = []
# for i in range(1,hspace_len):
#     jump_t1.append( np.sqrt(gamma_decay_new[i]) * qt.basis(hspace_len,0) * qt.basis(hspace_len,i).dag() )
#     jump_tphi.append( np.sqrt(2*gamma_dephase_new[i]) * qt.basis(hspace_len,i).proj() )


In [ ]:
# np.savez("data/data_xgate_theta_collapse_op.npz", arr1=c_op_list)

### Error

In [ ]:
U_loaded = qt.qload('U_noise_tg=55_truc=400')
kraus = qt.to_kraus(U_loaded, tol=1e-12)
kraus = [ut.truncate_2(i, [0,2]) for i in kraus]
# qt.metrics.average_gate_fidelity(kraus, target=qt.sigmax())
target=qt.sigmax()
d = kraus[0].shape[0]
ff = (d + np.sum([np.abs((A_k * target.dag()).tr())**2
                        for A_k in kraus])) / (d**2 + d)
ff

1.0000172511161765

In [ ]:
# U_loaded.trace_norm()

In [ ]:
U_loaded

Quantum object: dims = [[[199], [199]], [[2], [2]]], shape = (39601, 4), type = super, isherm = False
Qobj data =
  (0, 0)	(-1.2701060650832317e-05+1.0923744945433164e-13j)
  (0, 1)	(-0.00012079285755475356-4.901791387469262e-05j)
  (0, 2)	(-0.00012079285512918139+4.901786817197293e-05j)
  (0, 3)	(1.0000548101261764+6.166635667812724e-14j)
  (1, 0)	(3.8070743930443513e-07-5.534944103923757e-08j)
  (1, 1)	(-5.365980729202981e-08-1.1646178408215223e-07j)
  (1, 2)	(5.63534590948122e-05-0.000333638362962554j)
  (1, 3)	(1.4296147876529303e-05+0.0003379808011432542j)
  (2, 0)	(-0.00021254901871758835+4.313909253992107e-06j)
  (2, 1)	(-2.77606792865969e-05+2.022190063505309e-06j)
  (2, 2)	(0.9999470657836839-0.00022765759973680626j)
  (2, 3)	(0.0001216292448205111+1.5639310983811423e-06j)
  (3, 0)	(2.5729233403963866e-08+2.259456665334984e-08j)
  (3, 1)	(7.112849622337906e-08-1.2615564966789213e-07j)
  (3, 2)	(-4.82243680568138e-07-2.3173214248585088e-07j)
  (3, 3)	(2.9312282959766323e-07+1.6

In [ ]:
U_loaded

Quantum object: dims = [[[199], [199]], [[2], [2]]], shape = (39601, 4), type = super, isherm = False
Qobj data =
  (0, 0)	(-1.2701060650832317e-05+1.0923744945433164e-13j)
  (0, 1)	(-0.00012079285755475356-4.901791387469262e-05j)
  (0, 2)	(-0.00012079285512918139+4.901786817197293e-05j)
  (0, 3)	(1.0000548101261764+6.166635667812724e-14j)
  (1, 0)	(3.8070743930443513e-07-5.534944103923757e-08j)
  (1, 1)	(-5.365980729202981e-08-1.1646178408215223e-07j)
  (1, 2)	(5.63534590948122e-05-0.000333638362962554j)
  (1, 3)	(1.4296147876529303e-05+0.0003379808011432542j)
  (2, 0)	(-0.00021254901871758835+4.313909253992107e-06j)
  (2, 1)	(-2.77606792865969e-05+2.022190063505309e-06j)
  (2, 2)	(0.9999470657836839-0.00022765759973680626j)
  (2, 3)	(0.0001216292448205111+1.5639310983811423e-06j)
  (3, 0)	(2.5729233403963866e-08+2.259456665334984e-08j)
  (3, 1)	(7.112849622337906e-08-1.2615564966789213e-07j)
  (3, 2)	(-4.82243680568138e-07-2.3173214248585088e-07j)
  (3, 3)	(2.9312282959766323e-07+1.6

In [ ]:
U_loaded.tr()

(0.9999346041860553-0.0002276042683870902j)

In [ ]:
# truncate kraus operators
U_loaded = qt.qload('U_noise_tg=55_truc=400')
kraus = qt.to_kraus(U_loaded, tol=1e-12)
print(f'U_loaded.istp={U_loaded.istp}')
print(f'U_loaded.iscp={U_loaded.iscp}')
print('np.shape(kraus)=', np.shape(kraus))
ii = [i.dag()* i for i in kraus]
sum(ii)

U_loaded.istp=True
U_loaded.iscp=False
np.shape(kraus)= (383, 199, 2)


Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[1.04762417+0.j         0.00227454+0.00912945j]
 [0.00227454-0.00912945j 1.03978096+0.j        ]]

In [ ]:

kraus = [ut.truncate_2(i, [0,2]) for i in kraus]
super_op_post = qt.kraus_to_super(kraus)
fidelity = qt.metrics.average_gate_fidelity(super_op_post, target=qt.sigmax())
print(f'fidelity={fidelity:.8f}')
print(f'super_op_post.istp={super_op_post.istp}')
print(f'super_op_post.iscp={super_op_post.iscp}')
super_op_post.tidyup(atol=1e-2)
# qt.Qobj(U_loaded.tidyup(atol=1e-4)[398:408,:4])
# qt.to_super(qt.sigmax())

In [ ]:
ii = [i.dag()* i for i in kraus]
sum(ii)

Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[ 1.02733211+0.j         -0.0061129 +0.00477771j]
 [-0.0061129 -0.00477771j  1.00486537+0.j        ]]

In [ ]:
evals = []
ii = []
for i in kraus:
    ii.append(i.dag()* i)
    # evals.append(np.abs(i.eigenenergies()))
    print('evals=', np.abs((i.dag()* i).eigenenergies()))
# print('evals=', evals)
    # print('ii=', ii)

evals= [0.99978778 1.00026292]
evals= [8.51296061e-06 1.38860845e-02]
evals= [1.51749797e-05 1.38749803e-02]
evals= [0.00050932 0.0015055 ]
evals= [0.00047172 0.00143559]
evals= [1.59868533e-05 1.47637629e-04]
evals= [3.26770002e-05 1.16225693e-04]
evals= [8.34365793e-06 1.36107553e-05]
evals= [2.18162271e-07 4.06080422e-05]
evals= [8.75569612e-06 1.69298015e-05]
evals= [3.46994808e-07 3.17510465e-05]
evals= [6.86515664e-10 7.36002083e-09]
evals= [1.14162841e-07 4.58608858e-07]
evals= [4.63166629e-09 1.68403048e-06]
evals= [1.04514950e-07 2.95290452e-06]
evals= [2.30149945e-07 7.98402719e-07]
evals= [1.00982474e-07 2.83833468e-07]
evals= [3.67512642e-09 2.30297745e-08]
evals= [4.91711277e-09 2.61100030e-08]
evals= [2.07722984e-09 7.51080888e-09]
evals= [4.56790273e-10 2.62512183e-09]
evals= [9.65640967e-12 1.11584145e-10]
evals= [2.49084574e-11 2.77701916e-10]
evals= [2.61952501e-11 3.30713705e-10]
evals= [1.86120752e-11 1.16866865e-10]
evals= [1.30033598e-10 2.00896602e-09]
evals= [1.

In [ ]:
evals = []
ii = []
for i in kraus2:
    ii.append(i.dag()* i)
    evals.append(i.eigenenergies())
print('evals=', evals)
print('ii=', ii)

evals= [array([0.00000000e+00+0.06009088j, 1.73472348e-18-0.06009088j]), array([-1.00001294+0.j,  1.00001294+0.j]), array([0.        , 0.15381595])]
ii= [Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[0.00361078 0.        ]
 [0.         0.00361104]], Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[1.00006198 0.        ]
 [0.         0.99998977]], Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[0.02365935 0.        ]
 [0.         0.        ]]]


In [ ]:
kraus2 = qt.to_kraus(super_op_post)
kraus2

[Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = False
 Qobj data =
 [[ 0.          0.06009197]
  [-0.0600898   0.        ]],
 Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = False
 Qobj data =
 [[0.         0.99999489]
  [1.00003099 0.        ]],
 Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
 Qobj data =
 [[0.15381595 0.        ]
  [0.         0.        ]]]

In [ ]:
kraus

[Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = False
 Qobj data =
 [[-2.27858815e-04+1.17713159e-04j  9.99994165e-01+1.64682877e-04j]
  [ 1.00003114e+00+1.44148483e-08j  5.02440293e-05-3.56462085e-05j]],
 Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = False
 Qobj data =
 [[-0.08068451+0.07091276j  0.0091651 -0.03171995j]
  [-0.00816759+0.03195754j -0.00434205-0.01260896j]],
 Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = False
 Qobj data =
 [[ 0.07086866+0.08105097j -0.03124753-0.0086154j ]
  [ 0.03096082+0.00963774j -0.01337016+0.00420419j]],
 Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = False
 Qobj data =
 [[-0.00642344+0.0128601j  -0.00221418+0.02695265j]
  [ 0.00211331-0.02713812j  0.01821862+0.00199193j]],
 Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = False
 Qobj data =
 [[-0.01289037-0.00684102j -0.02568685-0.00099923j]
  [ 0.02548153+0.00107699j 

In [ ]:
np.shape(kraus2), np.shape(kraus)

((3, 2, 2), (208, 2, 2))

In [ ]:
U_ideal = qt.Qobj(np.array([[ 1.02611440e-04+4.80672891e-05j, -7.16719444e-01-6.97350021e-01j],
 [-7.16782423e-01-6.97285320e-01j, -8.02048805e-05-1.21720702e-04j]]))
qt.to_super(U_ideal).tidyup(atol=1e-4)

Quantum object: dims = [[[2], [2]], [[2], [2]]], shape = (4, 4), type = super, isherm = False
Qobj data =
[[ 0.00000000e+00 -1.07063339e-04 -1.07063339e-04  9.99983813e-01]
 [-1.07066692e-04  0.00000000e+00  9.99983832e-01  1.42366331e-04]
 [-1.07066692e-04  9.99983832e-01  0.00000000e+00  1.42366331e-04]
 [ 9.99983859e-01  1.42363507e-04  1.42363507e-04  0.00000000e+00]]

In [ ]:
U_loaded = qt.qload('U_noise_tg=55_truc=400')
U_loaded.tidyup(atol=1e-2)

Quantum object: dims = [[[199], [199]], [[2], [2]]], shape = (39601, 4), type = super, isherm = False
Qobj data =
  (0, 3)	(1.0000548101261764+0j)
  (2, 2)	(0.9999470657836839+0j)
  (198, 0)	-0.012962495633780343j
  (198, 1)	(0.01735445139115145-0.019084666325474228j)
  (198, 3)	0.011150838936642888j
  (398, 1)	(0.9999470657835142+0j)
  (400, 0)	(1.0001522038198687+0j)
  (1193, 1)	-0.014410376155626231j
  (39402, 0)	0.013056141546649035j
  (39402, 2)	(0.01735445155858764+0.019084666171498062j)
  (39402, 3)	-0.011152840189275034j
  (39407, 2)	0.014410376162454776j

In [ ]:
qt.Qobj(U_loaded.tidyup(atol=1e-4)[398:408,:4]).tidyup(atol=1e-2)


Quantum object: dims = [[10], [4]], shape = (10, 4), type = oper, isherm = False
Qobj data =
[[0.         0.99994707 0.         0.        ]
 [0.         0.         0.         0.        ]
 [1.0001522  0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]]

In [ ]:
U_loaded

Quantum object: dims = [[[199], [199]], [[2], [2]]], shape = (39601, 4), type = super, isherm = False
Qobj data =
  (0, 0)	(-1.2701060650832317e-05+1.0923744945433164e-13j)
  (0, 1)	(-0.00012079285755475356-4.901791387469262e-05j)
  (0, 2)	(-0.00012079285512918139+4.901786817197293e-05j)
  (0, 3)	(1.0000548101261764+6.166635667812724e-14j)
  (1, 0)	(3.8070743930443513e-07-5.534944103923757e-08j)
  (1, 1)	(-5.365980729202981e-08-1.1646178408215223e-07j)
  (1, 2)	(5.63534590948122e-05-0.000333638362962554j)
  (1, 3)	(1.4296147876529303e-05+0.0003379808011432542j)
  (2, 0)	(-0.00021254901871758835+4.313909253992107e-06j)
  (2, 1)	(-2.77606792865969e-05+2.022190063505309e-06j)
  (2, 2)	(0.9999470657836839-0.00022765759973680626j)
  (2, 3)	(0.0001216292448205111+1.5639310983811423e-06j)
  (3, 0)	(2.5729233403963866e-08+2.259456665334984e-08j)
  (3, 1)	(7.112849622337906e-08-1.2615564966789213e-07j)
  (3, 2)	(-4.82243680568138e-07-2.3173214248585088e-07j)
  (3, 3)	(2.9312282959766323e-07+1.6

In [ ]:
# truncate superoperator
n_truc = 199
U_truc = ut.truncate_2(U_loaded, [0,2, 2*n_truc+0, 2*n_truc+2])
U_super = qt.Qobj(U_truc, type='super')
fidelity = qt.metrics.average_gate_fidelity(U_super, target=qt.sigmax())
print(f'fidelity={fidelity:.8f}')
qt.Qobj(U_truc, type='super').tidyup(atol=1e-4)

fidelity=1.00001686


Quantum object: dims = [[[2], [2]], [[2], [2]]], shape = (4, 4), type = super, isherm = False
Qobj data =
[[0.         0.         0.         1.00005481]
 [0.         0.         0.99994707 0.        ]
 [0.         0.99994707 0.         0.        ]
 [1.0001522  0.         0.         0.        ]]

In [ ]:
U_super = 2*U_super / np.linalg.norm(U_super, 'fro')
fidelity = qt.metrics.average_gate_fidelity(U_super, target=qt.sigmax())
print(f'fidelity={fidelity:.8f}')
U_super

fidelity=1.00000000


In [ ]:
# truncate kraus operators
U_loaded = qt.qload('U_noise')
kraus = qt.to_kraus(U_loaded)
print(f'U_loaded.istp={U_loaded.istp}')
print(f'U_loaded.iscp={U_loaded.iscp}')
print('np.shape(kraus)=', np.shape(kraus))

kraus = [ut.truncate_2(i, [0,2]) for i in kraus]
super_op_post = qt.kraus_to_super(kraus)
fidelity = qt.metrics.average_gate_fidelity(super_op_post, target=qt.sigmax())
print(f'fidelity={fidelity:.8f}')
print(f'super_op_post.istp={super_op_post.istp}')
print(f'super_op_post.iscp={super_op_post.iscp}')
super_op_post.tidyup(atol=1e-2)
# qt.Qobj(U_loaded.tidyup(atol=1e-4)[398:408,:4])
# qt.to_super(qt.sigmax())

U_loaded.istp=True
U_loaded.iscp=False
np.shape(kraus)= (208, 199, 2)
fidelity=1.00001725
super_op_post.istp=False
super_op_post.iscp=True


Quantum object: dims = [[[2], [2]], [[2], [2]]], shape = (4, 4), type = super, isherm = False
Qobj data =
[[0.02365935 0.         0.         1.00360082]
 [0.         0.         0.99641496 0.        ]
 [0.         0.99641496 0.         0.        ]
 [1.00367276 0.         0.         0.        ]]

In [ ]:
U_super = 2* super_op_post / np.linalg.norm(super_op_post, 'fro')
fidelity = qt.metrics.average_gate_fidelity(U_super, target=qt.sigmax())
print(f'fidelity={fidelity:.8f}')
U_super

fidelity=0.99994901


Quantum object: dims = [[[2], [2]], [[2], [2]]], shape = (4, 4), type = super, isherm = False
Qobj data =
[[0.02365693 0.         0.         1.0034981 ]
 [0.         0.         0.99631298 0.        ]
 [0.         0.99631298 0.         0.        ]
 [1.00357004 0.         0.         0.        ]]

In [ ]:
print(f'super_op_post.istp={U_super.istp}')
print(f'super_op_post.iscp={U_super.iscp}')

super_op_post.istp=False
super_op_post.iscp=True


In [ ]:
d = 2  # new truncated space dimension
check_tp = sum([K.dag() * K for K in kraus])
print("Deviation from identity:", (check_tp - qt.qeye(d)).norm())

Deviation from identity: 0.032197484182331385


In [ ]:
from scipy.linalg import fractional_matrix_power

kraus = qt.to_kraus(qt.to_super(U_loaded))
kraus = [ut.truncate_2(i, [0,2]) for i in kraus]

# Compute K†K
KdagK = sum([K.dag() * K for K in kraus])

# Convert to dense array
KdagK_dense = KdagK.full()

# Compute inverse square root
inv_sqrt_dense = fractional_matrix_power(KdagK_dense, -0.5)

# Wrap back into Qobj
inv_sqrt = qt.Qobj(inv_sqrt_dense, dims=KdagK.dims)

# Apply to Kraus operators
kraus = [inv_sqrt * K for K in kraus]

# Build superoperator
super_op_post = qt.kraus_to_super(kraus)
print("TP?", super_op_post.istp)
print("Fidelity:", qt.metrics.average_gate_fidelity(super_op_post, target=qt.sigmax()))


TP? False
Fidelity: 0.9895430930776673


In [ ]:
print(f'super_op_post.istp={super_op_post.istp}')
print(f'super_op_post.iscp={super_op_post.iscp}')
print(f'fidelity={qt.metrics.average_gate_fidelity(super_op_post, target=qt.sigmax())}')
super_op_post

super_op_post.istp=False
super_op_post.iscp=True
fidelity=1.000017251116177


Quantum object: dims = [[[2], [2]], [[2], [2]]], shape = (4, 4), type = super, isherm = False
Qobj data =
[[ 2.36593472e-02+4.37265220e-20j -5.40259478e-03+3.36338513e-03j
  -5.40259478e-03-3.36338513e-03j  1.00360082e+00-1.85690229e-20j]
 [ 4.94629878e-03-3.59314123e-03j -1.33735268e-03+2.24726298e-03j
   9.96414962e-01-1.64343150e-04j  8.10942837e-04-1.48539735e-03j]
 [ 4.94629878e-03+3.59314123e-03j  9.96414962e-01+1.64343150e-04j
  -1.33735268e-03-2.24726298e-03j  8.10942837e-04+1.48539735e-03j]
 [ 1.00367276e+00-1.72117297e-20j -7.10305384e-04+1.41432520e-03j
  -7.10305384e-04-1.41432520e-03j  1.26455448e-03+6.14469370e-21j]]

In [ ]:
# drive_phi, drive_theta, truc = False, True, 150
drive_phi, drive_theta, truc = True, False, 150
drive_0 = True

t1_other = 170 # μs
gamma_decay_other =  1 / 1e3 / t1_other
gamma_dephase_other = 1 / 1e3 / t1_other

############################################################
folder = 'data_xgate_theta_3ncut.txt' if drive_theta else 'data_xgate_phi_3ncut.txt'
f_xgate = pd.read_csv('data/'+folder)
params = f_xgate[['tg', 'drive_amp_1', 'drive_amp_2', 'detune_1', 'detune_2'
                    ]].to_numpy()[1::4, :]
params

array([[3.00025120e+01, 2.22626000e-01, 2.01971000e-01, 3.89445000e-01,
        4.25332000e-01],
       [7.00010440e+01, 2.09788000e-01, 1.31851000e-01, 3.53189000e-01,
        3.69970000e-01],
       [1.10009928e+02, 2.11658000e-01, 1.06989000e-01, 3.41806000e-01,
        3.62874000e-01],
       [1.50002399e+02, 2.05268000e-01, 9.45520000e-02, 3.39607000e-01,
        3.55509000e-01],
       [1.90009275e+02, 2.09698000e-01, 8.34950000e-02, 3.35382000e-01,
        3.59146000e-01]])